# Adult GNN Ablation — Kaggle (GPU) — TabDiff XGBoost evaluation

Runs `ablation/ablation_runner.py` on the **adult** dataset using the **dual-split
design**:

- **This notebook** trains, samples, and evaluates entirely on the dedicated
  TabDiff-comparable split `data/adult_tabdiff_split/` (built by
  `scripts/make_tabdiff_splits.py`, seed 0, sizes matching TabDiff's Table 6),
  with **TabDiff's exact XGBoost protocol** (`ablation/eval_xgboost.py`, a
  faithful port of `eval/mle/mle.py` from MinkaiXu/TabDiff). Results land under
  `exp/adult_tabdiff_split/` so they are never conflated with the original-split results.
- The **original split** `data/adult/` stays untouched — it remains the one used
  for the CatBoost/TabDDPM-protocol comparison, with its own results under
  `exp/adult/`.

Downstream reporting keeps these as **two separate tables** (CatBoost vs TabDDPM,
XGBoost vs TabDiff) — never merged into one number.

**How to use**
1. In the notebook sidebar set **Accelerator → GPU** and **Internet → On**.
2. Run the setup cells (clone + install) once.
3. Edit only the **ABLATION PARAMETERS** cell to change what gets swept, then run the last cell.

These parameters mirror the module-level constants in `ablation_runner.py`
(`GNN_TYPES`, `N_LAYERS`, `D_MODELS`, `TOP_KS`, `N_HEADS`, `ATTENTIONS`,
`ABLATION_STEPS`, seed / diffusion constants). We override them in-memory so you
never have to touch the source file.


## 1. Check GPU


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))


## 2. Get the repo (+ data)

Default: clone the public GitHub repo (data is committed in the repo, ~0.5 GB).

If the repo is **private**, either:
- upload it as a Kaggle Dataset and set `REPO_SOURCE = 'kaggle'` + `KAGGLE_DATASET_DIR`, or
- put a GitHub token in `GIT_TOKEN` below.


In [ ]:
import os, shutil

REPO_SOURCE = 'git'                 # 'git'  or  'kaggle'
GIT_URL     = 'https://github.com/stefromp/thesis_project'
GIT_BRANCH  = 'main'
GIT_TOKEN   = ''                    # optional: personal access token for a private repo

# Only used when REPO_SOURCE == 'kaggle' (repo uploaded as a Kaggle Dataset)
KAGGLE_DATASET_DIR = '/kaggle/input/thesis-project'

REPO_ROOT = '/kaggle/working/thesis_project'

if REPO_SOURCE == 'git':
    if os.path.isdir(REPO_ROOT):
        shutil.rmtree(REPO_ROOT)
    url = GIT_URL
    if GIT_TOKEN:
        url = url.replace('https://', f'https://{GIT_TOKEN}@')
    !git clone --depth 1 --branch {GIT_BRANCH} {url} {REPO_ROOT}
elif REPO_SOURCE == 'kaggle':
    # Kaggle input is read-only, so copy it into the writable working dir.
    if os.path.isdir(REPO_ROOT):
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(KAGGLE_DATASET_DIR, REPO_ROOT)
else:
    raise ValueError('REPO_SOURCE must be "git" or "kaggle"')

# This notebook runs on the dedicated TabDiff split; the original split must
# also be present (untouched) — both ship with the repo.
assert os.path.isdir(os.path.join(REPO_ROOT, 'data', 'adult_tabdiff_split')), \
    'TabDiff split not found under REPO_ROOT/data/adult_tabdiff_split — run scripts/make_tabdiff_splits.py and commit it'
assert os.path.isdir(os.path.join(REPO_ROOT, 'data', 'adult')), \
    'original adult data not found under REPO_ROOT/data/adult'
print('REPO_ROOT =', REPO_ROOT)
print(sorted(os.listdir(REPO_ROOT)))


## 3. Install dependencies

We keep Kaggle's pre-installed GPU build of **torch / numpy / pandas / scipy /
scikit-learn** (downgrading them tends to break CUDA) and only add the extra
packages the eval pipeline needs. `xgboost` is added for TabDiff's evaluator, `sdmetrics` for the paper's
fidelity metrics (Shape / Trend / C2ST)
(Kaggle preinstalls a GPU-capable build, so this is usually a no-op).

We deliberately do **not** install `rtdl` / `libzero` — they are source-only on
recent Python and fail to build, and `ablation_runner` injects lightweight stubs
for the `zero` / `rtdl` modules automatically. Packages are installed unpinned so
pip can pick a wheel, and each on its own line so one failure can't abort the rest.


In [ ]:
import sys, importlib, subprocess

# (pip_name, import_name) for everything the pipeline imports.
DEPS = [
    ('icecream',          'icecream'),
    ('catboost',          'catboost'),
    ('category-encoders', 'category_encoders'),
    ('tomli',             'tomli'),
    ('tomli_w',           'tomli_w'),
    ('pynvml',            'pynvml'),
    ('skorch',            'skorch'),
    ('xgboost',           'xgboost'),
    ('sdmetrics',         'sdmetrics'),
]

def have(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

# Only install what's actually missing (Kaggle preinstalls most of these), and
# force a wheel so pip never falls back to a source build (the build failures).
missing = [(pip, mod) for pip, mod in DEPS if not have(mod)]
print('missing:', [m for _, m in missing] or 'none')

for pip_name, mod in missing:
    print(f'>>> installing {pip_name}')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', pip_name],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        # Fall back to allowing a source build, but show the real error if it fails.
        r2 = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name],
            capture_output=True, text=True,
        )
        if r2.returncode != 0:
            print(f'    !! FAILED {pip_name}\n{(r.stderr or r2.stderr)[-1500:]}')

importlib.invalidate_caches()
print('\nfinal check:')
for _, mod in DEPS:
    print('  ok  ' if have(mod) else '  FAIL', mod)


## 4. Adult dataset configuration (TabDiff split)

Base config pulled from the repo's registry
(`ablation/run_all_ablation.py` → `ALL_DATASETS['adult']`), then repointed at the
dedicated TabDiff split `data/adult_tabdiff_split/`. `num_samples` is set to the split's
train size, matching TabDiff's protocol of evaluating on a synthetic table the
same size as the real training set.

The cell **confirms the TabDiff split matches TabDiff's Table 6 sizes exactly** (it was
constructed to match, so this is an assert, not a warning), and states where the
untouched original split lives.


In [ ]:
import sys
import numpy as np

for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'ablation'), os.path.join(REPO_ROOT, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

from run_all_ablation import ALL_DATASETS

DATASET_NAME     = 'adult'
TABDIFF_SPLIT    = 'adult_tabdiff_split'          # data folder AND exp/ output name
EXPECTED_SIZES   = {'train': 28943, 'val': 3618, 'test': 16281}

CFG = dict(ALL_DATASETS[DATASET_NAME])
CFG['real_data_path'] = f'data/{TABDIFF_SPLIT}/'
CFG['num_samples']    = EXPECTED_SIZES['train']   # TabDiff: synthetic == train size
print(CFG)

# ---- confirm the TabDiff split matches the expected sizes exactly ----
data_dir = os.path.join(REPO_ROOT, CFG['real_data_path'])
sizes = {s: len(np.load(os.path.join(data_dir, f'y_{s}.npy'), allow_pickle=True))
         for s in ('train', 'val', 'test')}
assert sizes == EXPECTED_SIZES, f'TabDiff split size mismatch: {sizes} != {EXPECTED_SIZES}'
print('TabDiff split verified:', sizes)
print('Original split (CatBoost/TabDDPM protocol) is untouched at',
      os.path.join(REPO_ROOT, 'data', DATASET_NAME),
      '— its results live under exp/' + DATASET_NAME + '/, separate from this run.')


## 5. ABLATION PARAMETERS  ← edit this cell each run

These are exactly the constants you'd otherwise edit at the top of
`ablation_runner.py`. The full grid is the Cartesian product of the lists
(GCN/GIN ignore `N_HEADS`, so only the first head value runs for them).

Grid size = `len(GNN_TYPES) * len(N_LAYERS) * len(D_MODELS) * len(TOP_KS) * len(N_HEADS) * len(ATTENTIONS)`
(minus the redundant head combos for gcn/gin).


In [ ]:
# ---- ablation grid (change these) ----
GNN_TYPES  = ['gcn','gatv2','gin']   # subset of: gcn | gat | gatv2 | gin
N_LAYERS   = [2,3]                   # number of blocks
D_MODELS   = [32, 64]                # hidden dim
TOP_KS     = [3, 5]                  # sparsity_top_k (0 = dense)
N_HEADS    = [4]                     # attention heads (gcn/gin ignore)
ATTENTIONS = [False]                 # dense self-attention sublayer on/off

# ---- training / eval budget ----
ABLATION_STEPS         = 20_000
N_GEN_SEEDS            = 5
# TabDiff's XGBoost protocol is deterministic (fixed grid, no per-seed
# randomness once the val split is fixed), so 1 clf seed per gen seed is
# enough — extra seeds would just repeat identical evaluations.
N_CLF_SEEDS            = 1
ABLATION_NUM_TIMESTEPS = 1000
ABLATION_BATCH_SIZE    = 4096
ABLATION_LR            = 1e-4

# ---- run options ----
DEVICE  = 'cuda:0'
SEED    = 0
NO_SKIP = False   # True = re-run combos even if results already exist


## 6. Run the ablation (on the TabDiff split)

Imports the real `ablation_runner`, overwrites its module-level constants with
the values from the cell above, selects the **TabDiff XGBoost evaluator**
(`ar.EVALUATOR = 'xgboost'`) and the **TabDiff paper fidelity suite**
(`ar.TABDIFF_FIDELITY = True`: density Shape/Trend via sdmetrics, C2ST
detection, TabDiff DCR, naive α-precision/β-recall), then trains/samples/evaluates the full grid **on
the TabDiff split** — training data, validation, and test all come from
`data/adult_tabdiff_split/`. Results land in
`{REPO_ROOT}/exp/adult_tabdiff_split/ablation/<combo>/results_full_averaged.json` and a
consolidated `ablation_summary.json`, fully separate from the original-split
results in `exp/adult/`.


In [ ]:
import sys, argparse

for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'ablation'), os.path.join(REPO_ROOT, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

# eval_catboost loads tuned_models/catboost/<ds>_cv.json via a RELATIVE path,
# so the process cwd must be the repo root (harmless for xgboost, kept for parity).
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

import ablation_runner as ar

# Override the module constants with the values chosen above.
ar.GNN_TYPES             = GNN_TYPES
ar.N_LAYERS              = N_LAYERS
ar.D_MODELS              = D_MODELS
ar.TOP_KS                = TOP_KS
ar.N_HEADS               = N_HEADS
ar.ATTENTIONS            = ATTENTIONS
ar.ABLATION_STEPS        = ABLATION_STEPS
ar.N_GEN_SEEDS           = N_GEN_SEEDS
ar.N_CLF_SEEDS           = N_CLF_SEEDS
ar.ABLATION_NUM_TIMESTEPS = ABLATION_NUM_TIMESTEPS
ar.ABLATION_BATCH_SIZE   = ABLATION_BATCH_SIZE
ar.ABLATION_LR           = ABLATION_LR
ar.EVALUATOR             = 'xgboost'   # TabDiff's eval/mle protocol
ar.TABDIFF_FIDELITY      = True        # TabDiff paper suite: Shape/Trend, C2ST, DCR, alpha/beta

combos = list(ar.iter_ablation_grid())
print(f'Total combinations to run: {len(combos)}')
for c in combos:
    print('  ', ar.exp_dir_name(*c))

args = argparse.Namespace(
    repo_root=REPO_ROOT,
    data_root=REPO_ROOT,
    exp_root=REPO_ROOT,
    device=DEVICE,
    seed=SEED,
    skip_if_done=True,
    no_skip=NO_SKIP,
    aggregate=False,
)

# Passing TABDIFF_SPLIT as the dataset name routes every output (models,
# samples, metrics, summary) to exp/adult_tabdiff_split/ — never exp/adult/.
summary = ar.run_ablation(TABDIFF_SPLIT, CFG, args, single_combo=None)


## 7. Inspect / save results

Everything under `exp/adult_tabdiff_split/ablation/` is written into `/kaggle/working`, so
it is downloadable from the notebook's **Output** tab after the session ends.
ML-efficiency columns are TabDiff-style (macro-F1 / AUC); DCR / Wasserstein /
MIA-AUC are the TabDDPM-protocol metrics, unchanged.
Shape / Trend / C2ST / DCR / aPrec / bRec are the TabDiff-paper suite
(all averaged over generation seeds; DCR ~0.5 is ideal, the rest higher = better). This table is the
**TabDiff-split / XGBoost** result — keep it separate from the original-split
CatBoost table under `exp/adult/`.


In [ ]:
import json

summary_path = os.path.join(REPO_ROOT, 'exp', TABDIFF_SPLIT, 'ablation', 'ablation_summary.json')
with open(summary_path) as fh:
    data = json.load(fh)

print('summary:', summary_path, '\n')

def g(metrics, key):
    e = metrics.get(key) if metrics else None
    return f"{e['mean']:.4f}" if isinstance(e, dict) and 'mean' in e else '  -  '

# ML efficiency via TabDiff's XGBoost protocol (best-per-metric on val, scored
# on the TabDiff split's own test partition); plus the TabDDPM privacy metrics.
hdr = (f"{'combo':40s} {'status':8s} {'F1(macro)':>10s} {'AUC':>8s} "
       f"{'Shape':>7s} {'Trend':>7s} {'C2ST':>7s} {'DCR':>7s} "
       f"{'aPrec':>7s} {'bRec':>7s} "
       f"{'DCR_mean':>10s} {'Wass':>8s} {'MIA_AUC':>8s}")
print(hdr); print('-' * len(hdr))
for e in data:
    m = e.get('metrics', {})
    print(f"{e['exp_name']:40s} {e.get('status',''):8s} "
          f"{g(m,'macro avg/f1-score'):>10s} {g(m,'roc_auc'):>8s} "
          f"{g(m,'shape'):>7s} {g(m,'trend'):>7s} {g(m,'c2st'):>7s} {g(m,'dcr_tabdiff'):>7s} "
          f"{g(m,'alpha_precision'):>7s} {g(m,'beta_recall'):>7s} "
          f"{g(m,'dcr_tabddpm_mean'):>10s} {g(m,'wasserstein_mean'):>8s} "
          f"{g(m,'mia_auc'):>8s}")
